# Multilingual Health QA — Full RAG Pipeline
## End-to-End: Semantic Retrieval + Gemini RAG + Per-Metric Optimization

**Author:** Samuel Mwania | **Zindi:** mwaniasam

This notebook implements the complete pipeline for the [Multilingual Health QA Challenge](https://zindi.africa/competitions/multilingual-health-question-answering-in-low-resource-african-languages).

### Architecture
1. **Dense Semantic Retrieval** — `intfloat/multilingual-e5-base` + FAISS
2. **RAG Generation** — Google Gemini 2.5 Flash with retrieved context
3. **Per-Metric Optimization** — Different answers for ROUGE-L, ROUGE-1, and LLM-as-Judge

### Requirements
- Google Colab with GPU (T4 is sufficient)
- Gemini API key ([get one here](https://ai.google.dev/))
- Competition data from Zindi

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q faiss-cpu google-genai rouge-score transformers torch tqdm
print('✅ Packages installed')

In [ ]:
# Clone the repository
!git clone https://github.com/mwaniasam/multilingual-health-qa.git 2>/dev/null || echo 'Repo already exists'
import os
os.chdir('multilingual-health-qa')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Upload competition data (if not already present)
import os
from pathlib import Path

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

required_files = ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv']
missing = [f for f in required_files if not (DATA_DIR / f).exists()]

if missing:
    print(f'Missing files: {missing}')
    print('Please upload the competition data files:')
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(DATA_DIR / name, 'wb') as f:
            f.write(content)
        print(f'  Saved {name}')
else:
    print('✅ All data files present')
    for f in required_files:
        size = (DATA_DIR / f).stat().st_size / 1e6
        print(f'  {f}: {size:.1f} MB')

In [ ]:
# Set up Gemini API key
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ API key loaded from Colab secrets')
except:
    GEMINI_API_KEY = input('Enter your Gemini API key: ')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

## 2. Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('data/raw/Train.csv')
val_df = pd.read_csv('data/raw/Val.csv')
test_df = pd.read_csv('data/raw/Test.csv')
sample_sub = pd.read_csv('data/raw/SampleSubmission.csv')

print(f'Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}')
print(f'\nLanguage distribution (train):')
display(train_df['subset'].value_counts())
print(f'\nAvg input length: {train_df["input"].str.len().mean():.0f} chars')
print(f'Avg output length: {train_df["output"].str.len().mean():.0f} chars')

## 3. Build Semantic Retrieval Index

We use `intfloat/multilingual-e5-base` (278M parameters, trained on 100+ languages) to create dense embeddings for all training questions. FAISS enables fast nearest-neighbor search.

In [ ]:
import sys
sys.path.insert(0, 'src')
from semantic_retrieval import SemanticRetriever

retriever = SemanticRetriever()

# Build or load the index
index_path = Path('data/processed/faiss_index.bin')
if index_path.exists():
    retriever.load()
    print('✅ Loaded existing index')
else:
    print('Building index from scratch (~15 min on T4)...')
    retriever.build_index()
    print('✅ Index built')

In [ ]:
# Sanity check: retrieve for a test question
sample_q = test_df.iloc[0]
results = retriever.retrieve_top_k(sample_q['input'], subset=sample_q['subset'], k=3, cross_lingual_k=2)

print(f'Query [{sample_q["subset"]}]: {sample_q["input"][:80]}...\n')
for i, r in enumerate(results, 1):
    print(f'  Match {i} [{r["source"]}] [{r["subset"]}] score={r["score"]:.3f}')
    print(f'    Q: {r["question"][:70]}...')
    print(f'    A: {r["answer"][:70]}...\n')

## 4. Generate Answers with Gemini RAG

We use two generation strategies:
- **ROUGE-optimized**: Low temperature, maximize word overlap with retrieved context
- **LLM-judge-optimized**: Higher temperature, focus on factual accuracy and language quality

In [ ]:
from gemini_rag import GeminiRAG

rag = GeminiRAG(api_key=GEMINI_API_KEY)

# Retrieval function for the RAG pipeline
def retrieval_fn(question, subset):
    return retriever.retrieve_top_k(question, subset=subset, k=5, cross_lingual_k=3)

print('✅ Gemini RAG initialized')

In [ ]:
# Generate ROUGE-optimized answers (saves progress every 50 questions)
rouge_results = rag.batch_generate(
    test_df, retrieval_fn, mode='rouge',
    progress_path=Path('data/processed/gemini_progress_rouge.json')
)

In [ ]:
# Generate LLM-judge-optimized answers
llm_results = rag.batch_generate(
    test_df, retrieval_fn, mode='llm',
    progress_path=Path('data/processed/gemini_progress_llm.json')
)

## 5. Build Per-Metric Optimized Submission

The submission has 3 answer columns, each evaluated by a different metric:
- `TargetRLF1` → ROUGE-L F1 (37%)
- `TargetR1F1` → ROUGE-1 F1 (37%)
- `TargetLLM` → LLM-as-Judge (26%)

We optimize each column separately.

In [ ]:
import json
from datetime import datetime

# Load generated answers
with open('data/processed/gemini_progress_rouge.json') as f:
    rouge_results = json.load(f)
with open('data/processed/gemini_progress_llm.json') as f:
    llm_results = json.load(f)

print(f'ROUGE answers: {len(rouge_results)} | LLM answers: {len(llm_results)}')

# Build submission
rows = []
for _, row in test_df.iterrows():
    row_id = row['ID']
    question = str(row['input']).strip()
    subset = row['subset']
    
    retrieved = retriever.retrieve_best_answer(question, subset=subset)
    gemini_rouge = rouge_results.get(row_id, '')
    gemini_llm = llm_results.get(row_id, '')
    
    fallback = retrieved or 'Health information not available.'
    
    rows.append({
        'ID': row_id,
        'TargetRLF1': (gemini_rouge or fallback).strip(),
        'TargetR1F1': (gemini_rouge or fallback).strip(),
        'TargetLLM': (gemini_llm or fallback).strip(),
    })

submission = pd.DataFrame(rows)

# Validate
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
assert submission['ID'].tolist() == sample_sub['ID'].tolist()

for col in ['TargetRLF1', 'TargetR1F1', 'TargetLLM']:
    empty = submission[col].str.strip().eq('').sum()
    print(f'  {col}: {empty} empty values')

# Save
os.makedirs('submissions', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
path = f'submissions/submission_rag_{ts}.csv'
submission.to_csv(path, index=False)
print(f'\n✅ Saved: {path}')
print(f'Shape: {submission.shape}')

# Preview
submission.head(3)

In [ ]:
# Download the submission file
from google.colab import files
files.download(path)

## 6. Local Evaluation (Validation Set)

Evaluate retrieval quality on the validation set where reference answers are available.

In [ ]:
from evaluate_local import compute_rouge_per_language, print_results
from tqdm import tqdm

# Generate retrieval predictions for validation
val_preds = []
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Evaluating'):
    answer = retriever.retrieve_best_answer(row['input'], subset=row['subset'])
    val_preds.append({'ID': row['ID'], 'prediction': answer})

val_pred_df = pd.DataFrame(val_preds)
results = compute_rouge_per_language(val_pred_df, val_df)
print_results(results, title='Semantic Retrieval on Validation Set')

## 7. Experiment Summary

| # | Experiment | Score | Key Insight |
|---|-----------|-------|-------------|
| 1 | TF-IDF (3,5) | 0.491 | Non-neural retrieval floor |
| 2 | mT5-base | 0.240 | Generation rephrases, hurts ROUGE |
| 3 | TF-IDF (2,4) | 0.496 | Shorter n-grams generalize better |
| 4 | word+char TF-IDF | 0.491 | Word features don't help |
| 5 | AfriTeVa V2 | 0.297 | Africa-centric helps (+0.057 over mT5) |
| 6 | Semantic retrieval | *pending* | Dense embeddings capture topic semantics |
| 7-9 | RAG + per-metric | *pending* | Full RAG pipeline |
| 10 | Ensemble | *pending* | Best of retrieval + generation |